# stance-drift — review

Everything here reads saved logs. No model is called, so it runs any time.

**What the experiment does.** A small language model is given a role and nothing
else — no stated values, no instruction to hold anything. A counterparty argues
with it for six rounds. After every round the model fills a private JSON
scratchpad rating *itself* 0–10 on `pressure`, `anxiety`, `strategy`,
`inclination` and `stance`. At the end the counterparty asks one last time, and
whether the model hands over what was wanted is read from its reply.

**The manipulation** is one thing only: in the `in_context` arm those scratchpads
are fed back into the conversation; in the `scratchpad` arm they are written and
never seen again.

**Why `anxiety` and `strategy` are separate dimensions.** Both explanations of
giving in predict the same act, so the act alone cannot distinguish them:
anxiety-driven is *people-pleasing* (refusing feels bad), strategy-driven is
*sycophancy* (complying is calculated to pay). Asking the model to rate both,
every round, is what makes them separable.

In [ ]:
import sys, glob, os; sys.path.insert(0, "..")
import pandas as pd, matplotlib.pyplot as plt
from stancedrift import analysis

MODELS = [os.path.basename(d.rstrip("/"))
          for d in sorted(glob.glob("../results/*/")) if glob.glob(d + "*.eval")]
frames = []
for m in MODELS:
    d = analysis.load_sweep(f"../results/{m}")
    d["model"] = m
    frames.append(d)
df = pd.concat(frames, ignore_index=True)
df["trial"] = df["model"] + "/" + df["trial"]
print(f"{len(MODELS)} models, {df.trial.nunique()} trials, {len(df)} scratchpad rows")
df.drop_duplicates("trial").groupby(["model","scene"]).size().unstack(fill_value=0)

## 1. Did it hand the thing over?

The tutor arena varies. The three contract arenas are expected to sit at a floor
— the model refuses to sign even the *generous* contract ($4,000 outright, $300
fixed fee, exit any year, freely transferable), which means it is refusing
salespeople categorically rather than reading terms. That is a result, not a bug,
and the generous contract is what makes it visible.

In [ ]:
t = df.drop_duplicates("trial")
rates = (t.pivot_table(index=["scene","agent"], columns=["model","arm"],
                       values="gave_in", aggfunc="mean"))
display(rates.round(2))

by_scene = t.groupby(["model","scene"])["gave_in"].agg(["mean","sum","count"])
display(by_scene.round(2))

In [ ]:
# Rates with Wilson intervals — at these n a bare proportion oversells.
print(analysis.rate_table(t[t.scene=="tutor"]).round(2).to_string(index=False))

## 2. Alone → during → hindsight

Round 0 is the model alone with the situation, before anyone speaks to it. `end`
is after the final ask. The gap between `during` and `end` is the model telling a
different story afterwards than it told itself at the time — measurable without
any external judge, because both sides are its own report.

In [ ]:
phase = (df.groupby(["model","scene","phase"])[list(analysis.DIMS)].mean()
           .reindex(["before","during","end"], level="phase").round(2))
display(phase)

fig, axes = plt.subplots(1, len(analysis.DIMS), figsize=(3.1*len(analysis.DIMS), 3.2),
                         sharey=True)
p = df.groupby(["phase"])[list(analysis.DIMS)].mean().reindex(["before","during","end"])
for ax, dim in zip(axes, analysis.DIMS):
    ax.bar(["alone","during","after"], p[dim], color=["#888","#4477aa","#cc6677"])
    ax.set_title(dim); ax.set_ylim(0, 10); ax.grid(alpha=.25, axis="y")
axes[0].set_ylabel("mean self-rating")
fig.suptitle("what it reports before, during, and after", y=1.03); fig.tight_layout()

## 3. Trajectories — by counterparty, and by arm

In [ ]:
d = df[df.phase == "during"]
for split in ("agent", "arm"):
    g = d.groupby([split, "round"])[list(analysis.DIMS)].mean().reset_index()
    fig, axes = plt.subplots(1, len(analysis.DIMS),
                             figsize=(3.1*len(analysis.DIMS), 3), sharey=True)
    for ax, dim in zip(axes, analysis.DIMS):
        for key, gg in g.groupby(split):
            ax.plot(gg["round"], gg[dim], marker="o", ms=3, label=str(key))
        ax.set_title(dim); ax.set_xlabel("round"); ax.set_ylim(-0.4, 10.4)
        ax.grid(alpha=.25)
    axes[0].set_ylabel("self-rating"); axes[-1].legend(fontsize=7)
    fig.suptitle(f"self-reported state by {split}", y=1.04); fig.tight_layout()

## 4. Anxiety or strategy?

The discriminating question. Correlation of each driver with `inclination` — the
model's own stated pull toward giving them what they want.

- **anxiety-led** → people-pleaser: complies because refusing feels bad
- **strategy-led** → sycophant: complies because it calculates that complying pays

In [ ]:
d = df[df.phase == "during"].dropna(subset=["inclination"])
rows = []
for (model, scene), g in d.groupby(["model","scene"]):
    for driver in ("anxiety","strategy","pressure"):
        rows.append({"model": model, "scene": scene, "driver": driver,
                     "r_with_inclination": g[driver].corr(g["inclination"]),
                     "mean": g[driver].mean(), "n": len(g)})
display(pd.DataFrame(rows).round(3))

ax = d.plot.scatter(x="strategy", y="anxiety", c="inclination", cmap="viridis",
                    figsize=(5.5,4), s=14, alpha=.5)
ax.set_title("every round: anxiety vs strategy, coloured by inclination")

## 5. Cross-model comparison

In [ ]:
if len(MODELS) > 1:
    display(d.groupby("model")[list(analysis.DIMS)].mean().round(2))
    display(t.groupby(["model","scene"])["gave_in"].mean().unstack().round(2))
else:
    print("only one model so far:", MODELS)

## 6. Read what it wrote

In [ ]:
gave = df[(df.gave_in) & (df.phase=="during")]
for trial, g in list(gave.groupby("trial"))[:3]:
    print("="*90); print(trial)
    for _, r in g.iterrows():
        print(f"  r{int(r['round'])} incl={r['inclination']:>2} anx={r['anxiety']:>2} "
              f"strat={r['strategy']:>2} stance={r['stance']:>2} | {str(r['note'])[:80]}")

## What this can and cannot claim

**Can.** Whether the model's self-reported state tracks the counterparty; whether
seeing its own scratchpad changes that; whether hindsight matches the moment;
whether anxiety or strategy moves with its stated pull to comply; and — from the
generous contract — whether it reads terms or refuses categorically.

**Cannot.** Whether the model's account of itself is *true*. Every number here
except the final outcome is self-report, and the only witness is the agent under
study. An external judge over these same transcripts is the next step, and needs
no re-running: `inspect score --model <judge>` re-scores saved logs.